# Formula 1 Driver Performance Index (5 Seasons)

## Project aim
Build a composite **F1 Driver Performance Index** that compares drivers across the **last 5 completed seasons**.

## Notebook structure
1. Setup and imports
2. Download race and qualifying data for 5 seasons
3. Build clean event-level datasets
4. Aggregate to driver-season level
5. Handle missing data
6. Multivariate analysis
7. Normalisation
8. Weighting and aggregation
9. Compare with official standings
10. Visualisation
11. Export final outputs
12. Reflection, struggles, references, and commit log


In [1]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Data Setup

In [3]:
BASE_URL = 'https://api.jolpi.ca/ergast/f1'
SEASONS = [2021, 2022, 2023, 2024, 2025]

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_DIR = Path('outputs')

for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEASONS

[2021, 2022, 2023, 2024, 2025]

## 2. Download data

In [4]:
def fetch_json(endpoint: str) -> dict[str, Any]:
    url = f'{BASE_URL}/{endpoint}'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def get_rounds_for_season(season: int) -> list[int]:
    data = fetch_json(f'{season}.json')
    races = data['MRData']['RaceTable']['Races']
    return [int(r['round']) for r in races]